### Initialization and Environment Setup

This cell initializes the AutoREACTER environment and prepares the working directory for the current run.

In [ ]:
import json
from AutoREACTER.session import read_input
from AutoREACTER.input_parser import InputParser
from AutoREACTER.detectors.functional_groups_detector import FunctionalGroupsDetector
from AutoREACTER.detectors.reaction_detector import ReactionDetector
from AutoREACTER.detectors.non_monomer_detector import NonReactantsDetector
from AutoREACTER.reaction_preparation.reaction_processor.prepare_reactions import PrepareReactions
from AutoREACTER.reaction_preparation.ff_wrapper.molecule_3d_preparation import Molecule3DPreparation
from AutoREACTER.reaction_preparation.ff_wrapper.ff_wrapper import FFWrapper
from AutoREACTER.reaction_preparation.ff_wrapper.REACTER_files_builder import REACTERFilesBuilder
from AutoREACTER.sim_setup.simulation_setup import SimulationSetupManager

session = read_input("example_1_inputs_count_mode.json")

### Visualize Monomers (Optional)

In [ ]:
# Generate and display the initial monomer structures.
input_parser = InputParser()
initial_molecules = input_parser.initial_molecules_image_grid(session.inputs)
initial_molecules

### Functional Group Detection

This cell runs the functional group detection step.


In [ ]:
# Detect functional groups to get valid monomers

functional_groups_detector = FunctionalGroupsDetector()

functional_groups, functional_groups_imgs = (
    functional_groups_detector.functional_groups_detector(
        session.inputs.monomers
    )
)

### Functional Group Visualization (Optional)

This cell visualizes the detected functional groups on the molecules.

In [ ]:
# The resulting image shows each molecule with the detected functional groups marked.

img = functional_groups_detector.functional_group_highlighted_molecules_image_grid(functional_groups_imgs)
img

### Reaction Detection

This cell identifies possible reactions between the detected functional groups.


In [ ]:
# Detect possible reactions based on identified functional groups.

reaction_detector = ReactionDetector()
reaction_instances = reaction_detector.reaction_detector(functional_groups)

### Reaction visualization (Optional)

In [ ]:
# Visualize detected reactions.

img = reaction_detector.available_reaction_image_grid(reaction_instances)
img

### Reaction Selection

This cell filters and selects the relevant reactions from the detected reaction instances.


In [ ]:
# Select relevant reactions for template generation as user intened.

selected_reactions = reaction_detector.reaction_selection(reaction_instances)

### Non-Reactant (Non-Monomer) Detection

This cell identifies molecules that do **not participate in any detected reactions**.



In [ ]:
# Identify non-reactive molecules based on the selected reactions.

non_monomer_detector = NonReactantsDetector()

non_reactants_list = non_monomer_detector.non_monomer_detector(
    session.inputs,  
    selected_reactions
)

### Non reactants molecules visualization

In [ ]:
# Visualize non-reactive molecules (if any are detected).

img_non_reactants = non_monomer_detector.non_reactants_to_visualization(non_reactants_list)
img_non_reactants

### Update Inputs After Non-Reactant Filtering

This cell updates the validated inputs after identifying non-reactive molecules.


In [ ]:
# Update inputs by filtering out non-reactive molecules.

updated_inputs = non_monomer_detector.non_reactant_selection(
    session.inputs,
    non_reactants_list
)

### Prepare reaction templates from the selected reactions for downstream processing.

In [ ]:
prepare_reactions = PrepareReactions(session)
prepared_reactions = prepare_reactions.prepare_reactions(selected_reactions)

### Reaction Template Visualization

Visualize reaction templates with different highlighting options setting the `highlight_type` parameter to one of the following values:

- **Default** or **template**: Highlights all structural changes in the reaction templates  
- **edge**: Highlights edge atoms of the templates  
- **delete**: Highlights removed components (if applicable)  
- **initiators**: Highlights reaction initiator atoms  

In [ ]:
img = prepare_reactions.reaction_templates_highlighted_image_grid(prepared_reactions, highlight_type="edge")
img

### 3D Geometry Preparation

Prepare 3D molecular geometries for both inputs molecules and reaction templates.  
This step generates the structures required for the Lunar atom typing.

In [ ]:
molecule3dpreparation = Molecule3DPreparation(session)

updated_inputs_with_3d_mols, prepared_reactions_with_3d_mols = molecule3dpreparation.prepare_molecule_3d_geometry(
        updated_inputs, prepared_reactions
    )


### Lunar Workflow

Run the Lunar workflow to generate simulation-ready files from prepared 3D structures.

In [ ]:
ff_wrapper = FFWrapper(session)

ff_wrapper_results = ff_wrapper.generate_force_field_files(
    updated_inputs=updated_inputs_with_3d_mols,
    prepared_reactions=prepared_reactions_with_3d_mols
)

### REACTER File Generation and Finalization

Generate REACTER-compatible files from the Lunar results and prepared reactions.  
The generated files are then moved to the final run directory, and all internal paths are updated accordingly.

In [ ]:
from AutoREACTER.cache import RunDirectoryManager

builder = REACTERFilesBuilder(
    session=session,
    updated_inputs_with_3d_mols=updated_inputs_with_3d_mols,
)

reacter_files = builder.molecule_template_preparation(
    ff_files=ff_wrapper_results,
    prepared_reactions_with_3d_mols=prepared_reactions_with_3d_mols
)

# Move generated files to final output directory using RunDirectoryManager
run_manager = RunDirectoryManager(session.output_dir.parent)
reacter_files = run_manager.move_reacter_files(
    reacter_files,
    staging_dir=session.staging_dir,
    final_dir=session.output_dir
)


print(f"[OK] REACTER files successfully moved to {session.output_dir}")

### Simulation Setup and LAMMPS Script Generation

Calculate physical system properties (e.g., box sizes, required monomer counts to reach target densities) and generate the complete suite of LAMMPS input scripts. 

This automatically writes the configuration files for all 5 stages:
1. Densification
2. Pre-reaction Equilibration
3. First Reaction Stage (3.5A)
4. Second Reaction Stage (5.0A)
5. Post-reaction Equilibration

In [ ]:
Simulation_setup_manager = SimulationSetupManager()

updated_inputs_3d = Simulation_setup_manager.setup_and_write_simulation(
    setup=updated_inputs_with_3d_mols,
    reacter_files=reacter_files,
    run_dir=session.output_dir
)

print("\n[INFO] AutoREACTER workflow completed successfully.\n")
print(f"Final REACTER and LAMMPS files are located in: {session.output_dir}")